### Data ingestion to vector DB convertion


In [1]:
%pip install langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


C:\Users\rushi\AppData\Local\Temp\ipykernel_17724\1108623348.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader


In [3]:
import pymupdf
from pathlib import Path

for f in Path("../data").glob("**/*.pdf"):
    print(f"\n{f.name}  ({f.stat().st_size / 1e6:.1f} MB)")
    try:
        doc = pymupdf.open(f)
        empty = sum(1 for p in doc if not p.get_text().strip())
        print(f"  pages: {len(doc)}, empty pages: {empty}")
    except Exception as e:
        print(f"  cannot open: {e}")


AI & ML DIGITAL NOTES.pdf  (3.6 MB)
  pages: 177, empty pages: 0

andrew-ng-machine-learning-yearning.pdf  (5.7 MB)
  pages: 118, empty pages: 1

Linux_Commands_for_AI_Engineer_LEARNTECH_RUSHI.pdf  (2.1 MB)
  pages: 11, empty pages: 0

prepnplaced-data-engineering-book.pdf  (1.5 MB)
  pages: 244, empty pages: 0

sql.pdf  (0.5 MB)
  pages: 165, empty pages: 0


In [4]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

def load_pdf(pdf_file):
    """Try PyPDF first, fall back to PyMuPDF for damaged files."""
    for loader_cls in (PyPDFLoader, PyMuPDFLoader):
        try:
            docs = loader_cls(str(pdf_file)).load()
            # Treat "loaded but empty" as a failure too
            if sum(len(d.page_content.strip()) for d in docs) > 0:
                return docs, loader_cls.__name__
        except Exception as e:
            print(f"  {loader_cls.__name__} failed: {e}")
    return [], None


def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_files = list(Path(pdf_directory).glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        documents, used = load_pdf(pdf_file)

        if not documents:
            print("  X Could not extract text from this file")
            continue

        for doc in documents:
            doc.metadata["source_file"] = pdf_file.name
            doc.metadata["file_type"] = "pdf"

        total_chars = sum(len(d.page_content) for d in documents)
        all_documents.extend(documents)
        print(f"  Loaded {len(documents)} pages ({total_chars} characters) using {used}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents


all_pdf_documents = process_all_pdfs("../data")

Found 5 PDF files to process

Processing: AI & ML DIGITAL NOTES.pdf
  Loaded 177 pages (209619 characters) using PyPDFLoader

Processing: andrew-ng-machine-learning-yearning.pdf
  Loaded 118 pages (156759 characters) using PyPDFLoader

Processing: Linux_Commands_for_AI_Engineer_LEARNTECH_RUSHI.pdf
  Loaded 11 pages (7289 characters) using PyPDFLoader

Processing: prepnplaced-data-engineering-book.pdf
  Loaded 244 pages (395674 characters) using PyPDFLoader

Processing: sql.pdf
  Loaded 165 pages (74985 characters) using PyPDFLoader

Total documents loaded: 715


In [5]:
all_pdf_documents

[Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2023-01-04T05:56:26+00:00', 'source': '..\\data\\text_files\\AI & ML DIGITAL NOTES.pdf', 'total_pages': 177, 'page': 0, 'page_label': '1', 'source_file': 'AI & ML DIGITAL NOTES.pdf', 'file_type': 'pdf'}, page_content='ARTIFICIAL INTELLIGENCE \n& \n MACHINE LEARNING \n          Lecture Notes \nB.TECH \n(III YEAR – II SEM) \n(2022-2023) \nPrepared by:  \n                        Ms.Anitha Patibandla, Associate Professor \n                  Dr.B.Jyothi, Professor \n                                                    Ms.K.Bhavana, Assistant Professor \n \n \n \nMALLA REDDY COLLEGE OF ENGINEERING & TECHNOLOGY \n(Autonomous Institution – UGC, Govt. of India) \nDepartment of Electronics and Communication Engineering \n \nRecognized under 2(f) and 12 (B) of UGC ACT 1956 \n(Affiliated to JNTUH, Hyderabad, Approved by AICTE-Accredited by NBA &NAAC–‘A’Grade-ISO9001:2015 \nCertified) Maisammaguda,Dhulapa

### Chunking part


In [6]:
def split_documents(documents,chunk_size =1000,chunk_overlap =200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap =chunk_overlap,
        length_function =len,
        separators=["\n\n","\n"," ",""]
    )
    
    split_docs = text_splitter.split_documents(documents)
    print(f"Split{len(documents)} documents into {len(split_docs)} chunks")
    
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content:{split_docs[0].page_content[:20]}...")
        print(f"Metadata: {split_docs[0].metadata}")
        
    return split_docs    

In [7]:
chunks =split_documents(all_pdf_documents)
chunks

Split715 documents into 1279 chunks

Example chunk:
Content:ARTIFICIAL INTELLIGE...
Metadata: {'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2023-01-04T05:56:26+00:00', 'source': '..\\data\\text_files\\AI & ML DIGITAL NOTES.pdf', 'total_pages': 177, 'page': 0, 'page_label': '1', 'source_file': 'AI & ML DIGITAL NOTES.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2023-01-04T05:56:26+00:00', 'source': '..\\data\\text_files\\AI & ML DIGITAL NOTES.pdf', 'total_pages': 177, 'page': 0, 'page_label': '1', 'source_file': 'AI & ML DIGITAL NOTES.pdf', 'file_type': 'pdf'}, page_content='ARTIFICIAL INTELLIGENCE \n& \n MACHINE LEARNING \n          Lecture Notes \nB.TECH \n(III YEAR – II SEM) \n(2022-2023) \nPrepared by:  \n                        Ms.Anitha Patibandla, Associate Professor \n                  Dr.B.Jyothi, Professor \n                                                    Ms.K.Bhavana, Assistant Professor \n \n \n \nMALLA REDDY COLLEGE OF ENGINEERING & TECHNOLOGY \n(Autonomous Institution – UGC, Govt. of India) \nDepartment of Electronics and Communication Engineering \n \nRecognized under 2(f) and 12 (B) of UGC ACT 1956 \n(Affiliated to JNTUH, Hyderabad, Approved by AICTE-Accredited by NBA &NAAC–‘A’Grade-ISO9001:2015 \nCertified) Maisammaguda,Dhulapa

In [8]:
### Embedding and vector storage DB
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity


d:\RAG_projecty\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
class EmbeddingManager:
    """Handle document embedding generation using sentence-transformers."""

    def __init__(self, model_name: str = "all-MiniLM-L6-V2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the embedding model once and keep it in memory."""
        try:
            print(f"Loading embedding model:{self.model_name}")
            self.model =SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension:{self.model.get_sentence_embedding_dimension()}")
        
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self,texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of text

        Args:
            texts (List[str]): _description_

        Returns:
            np.ndarray: _description_
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape:{embeddings.shape}")
        return embeddings
    
## initialize the embedding manager

embedding_manager =EmbeddingManager()
embedding_manager   
        
        
    

Loading embedding model:all-MiniLM-L6-V2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1633.25it/s]


Model loaded successfully. Embedding dimension:384


C:\Users\rushi\AppData\Local\Temp\ipykernel_17724\3750944351.py:14: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension:{self.model.get_sentence_embedding_dimension()}")


In [10]:
import os
import hashlib
from typing import List, Any

import numpy as np
import chromadb


class VectorStore:

    def __init__(self, collection_name: str = "pdf_documents",
                 persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"},
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray, batch_size: int = 500):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        ids, metadatas, texts, embs = [], [], [], []
        for i, (doc, emb) in enumerate(zip(documents, embeddings)):
            # Same content + source + page -> same ID, so re-running doesn't duplicate
            key = f"{doc.metadata.get('source_file')}|{doc.metadata.get('page')}|{doc.page_content}"
            ids.append(hashlib.md5(key.encode()).hexdigest())

            # Keep only types Chroma accepts (drops None and nested values)
            meta = {k: v for k, v in doc.metadata.items()
                    if isinstance(v, (str, int, float, bool))}
            meta["doc_index"] = i
            meta["content_length"] = len(doc.page_content)
            metadatas.append(meta)

            texts.append(doc.page_content)
            embs.append(emb.tolist())

        try:
            for s in range(0, len(ids), batch_size):
                e = s + batch_size
                self.collection.upsert(
                    ids=ids[s:e],
                    embeddings=embs[s:e],
                    metadatas=metadatas[s:e],
                    documents=texts[s:e],
                )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise


vectorstore = VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 1279


In [11]:
for name in ["all_pdf_documents", "chunks", "embedding_manager", "vectorstore"]:
    print(name, "->", "OK" if name in globals() else "MISSING")

all_pdf_documents -> OK
chunks -> OK
embedding_manager -> OK
vectorstore -> OK


In [12]:
# convert the text to embeddings
texts = [doc.page_content for doc in chunks]

# generate the embeddings
embeddings = embedding_manager.generate_embeddings(texts)

# store chunks + embeddings in the vector database
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 1279 texts...


Batches:   0%|          | 0/40 [00:00<?, ?it/s]

Batches: 100%|██████████| 40/40 [01:42<00:00,  2.55s/it]


Generated embeddings with shape:(1279, 384)
Adding 1279 documents to vector store...
Successfully added 1279 documents to vector store
Total documents in collection: 1279


## RAG Retriever

In [13]:
class RAGRetriver:
    """Handles query-based retrivel from the vector store"""
    
    def __init__(self,vector_store:VectorStore,embedding_manager:EmbeddingManager):
        """_summary_

        Args:
            vector_store (VectorStore): _description_
            embedding_manager (EmbeddingManager): _description_
        """
        self.vectorstore = vector_store
        self.embedding_manager = embedding_manager
        
        
    def retrieve(self,query: str,top_k: int =5,score_threshold:float =0.0) -> List[Dict[str,Any]]:
        """
        Retrieve relevant documents for a query
        """    
        
        print(f"Retrieving documents for query: '{score_threshold}'")
        print(f"Top k:{top_k},Score threshold:{score_threshold}")
        
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        #search in the vector
        try:
            results = self.vectorstore.collection.query(
                query_embeddings =[query_embedding.tolist()],
                n_results =top_k
            )
            
            #Process results
            retrieved_docs =[]
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'] [0]
                ids = results['ids'][0]
                
                for i,(doc_id,document,metadata,distance) in enumerate(zip(ids,documents,metadatas,distances)):
                    #convert distance to similarity score (chromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id':doc_id,
                            'content':document,
                            'metadata':metadata,
                            'similarity_score':similarity_score,
                            'distance':distance,
                            'rank': i + 1
                        })  
                
                print(f"Retrived {len(retrieved_docs)}documents (after filtering)")
            else:
                print("No documents found")
                
            
            return retrieved_docs       
        
        except Exception as e:
            print(f"Error during the retrival:{e}")
            return []

rag_retriever =RAGRetriver(vectorstore,embedding_manager)       
                         
        

In [14]:
rag_retriever

In [15]:
rag_retriever.retrieve("What is python")

Retrieving documents for query: '0.0'
Top k:5,Score threshold:0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 31.37it/s]

Generated embeddings with shape:(1, 384)
Retrived 1documents (after filtering)


[{'id': '269e4617aa5ee6580409b6d4eeba6720',
  'content': '4C H A P T E R  F O U R\nPython for Data Engineers\nThis is noT a PyThon course.  A Python course teaches you\nclasses, decorators, generators, metaclasses and sorting\nalgorithms, and after four months you still cannot load a file into a\ndatabase without it breaking on Tuesday. This chapter teaches the\nsubset of Python that a data engineer uses, in the order the job\nuses it, and it ends with a working ingestion script you can run.\nVERSIONS AND DIALECTS\nPython  in  this  chapter  is  Python  3 standard  library,  plus  requests, \npandas and sqlite3. It was executed as printed.\nEverything here was executed. Every output block is what actually\nappeared on the terminal.\nHere is the shape of what we are building towards, and it is the\nshape of most data engineering code you will ever write:\nFIGURE 4.1 \xa0 The anatomy of nearly every ingestion script. Chapter 8 turns\nthis into a pipeline; Chapter 10 puts it on a schedule

In [16]:
rag_retriever.retrieve("What is Bias and varience")

Retrieving documents for query: '0.0'
Top k:5,Score threshold:0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 17.71it/s]

Generated embeddings with shape:(1, 384)
Retrived 1documents (after filtering)


[{'id': 'b11150c4d7276e8dca2a24cf0f61ac93',
  'content': 'Bias and Variance \n \n \n \n \n \n \xa0\nPage 41 Machine Learning Yearning-Draft Andrew Ng',
  'metadata': {'total_pages': 118,
   'source_file': 'andrew-ng-machine-learning-yearning.pdf',
   'creationdate': '2022-03-02T11:13:23+06:00',
   'moddate': '2022-03-02T11:13:23+05:00',
   'source': '..\\data\\text_files\\andrew-ng-machine-learning-yearning.pdf',
   'page_label': '41',
   'title': 'andrew-ng-machine-learning-yearning',
   'page': 40,
   'producer': 'Adobe PDF library 16.03',
   'content_length': 81,
   'file_type': 'pdf',
   'creator': 'Adobe Illustrator 26.0 (Windows)',
   'doc_index': 392},
  'similarity_score': 0.02885991334915161,
  'distance': 0.9711400866508484,
  'rank': 1}]

## integration of the Vectordb Contexr pipeline with LLM output

In [17]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv, find_dotenv

print(".env found at:", find_dotenv() or "NOT FOUND")
load_dotenv(find_dotenv(), override=True)

key = os.getenv("GROQ_API_KEY") or os.getenv("GROQ_API_KEY")
print("key loaded:", bool(key), "| length:", len(key) if key else 0)

.env found at: d:\RAG_projecty\notebook\.env
key loaded: True | length: 56


In [18]:
import os
print(os.getcwd())

d:\RAG_projecty\notebook


In [30]:
### RAG pipeline using Groq LLM
import os
from dotenv import load_dotenv, find_dotenv
from langchain_groq import ChatGroq

load_dotenv(find_dotenv(usecwd=True), override=True)

groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise ValueError("GROQ_API_KEY not found. Check your .env file.")

llm_model = ChatGroq(
    api_key=groq_api_key,
    model="openai/gpt-oss-20b",   # verify against the list below
    temperature=0.1,
    max_tokens=1024,
)

# Simple RAG function: retrieve context + generate response
def rag_simple(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc["content"] for doc in results]) if results else ""
    if not context:
        return "No relevant documents found to answer the query."

    prompt = f"""Use the following context to answer the question concisely.

Context:
{context}

Question: {query}

Answer:"""

    response = llm.invoke(prompt)
    return response.content

In [31]:
answer = rag_simple("What is Addressing bias?",rag_retriever,llm_model)
print(answer)

Retrieving documents for query: '0.0'
Top k:3,Score threshold:0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.65it/s]


Generated embeddings with shape:(1, 384)
Retrived 0documents (after filtering)
No relevant documents found to answer the query.


In [32]:
print(rag_simple(query="what is bias", retriever=rag_retriever, llm=llm_model))

Retrieving documents for query: '0.0'
Top k:3,Score threshold:0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 20.43it/s]

Generated embeddings with shape:(1, 384)
Retrived 2documents (after filtering)


Bias is the algorithm’s systematic error – roughly the error rate it would have on the training set if the training set were very large. In other words, it’s the training‑set error that reflects how far the model’s predictions are from the true underlying pattern.


# Enhanceded RAG Pipeline Features

In [36]:


# Enhanced RAG Pipeline Features --
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)

    if not results:
        return {
            "answer": "No relevant context found.",
            "sources": [],
            "confidence": 0.0,
            "context": ""
        }

    context = "\n\n".join([doc["content"] for doc in results])
    sources = [
        {
            "source": doc["metadata"].get("source_file", doc["metadata"].get("source", "unknown")),
            "page": doc["metadata"].get("page", "unknown"),
            "score": doc["similarity_score"],
            "preview": doc["content"][:300] + "..."
        }
        for doc in results
    ]
    confidence = max(doc["similarity_score"] for doc in results)

    prompt = f"""Use the following context to answer the question concisely.

Context:
{context}

Question: {query}

Answer:"""

    response = llm.invoke(prompt)

    output = {
        "answer": response.content,
        "sources": sources,
        "confidence": confidence
    }

    if return_context:
        output["context"] = context

    return output

result = rag_advanced("what is SQL?", rag_retriever, llm_model, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result["answer"])
print("Sources:", result["sources"])
print("Confidence:", result["confidence"])
print("Context Preview:", result["context"][:300])

Retrieving documents for query: '0.1'
Top k:3,Score threshold:0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.98s/it]


Generated embeddings with shape:(1, 384)
Retrived 3documents (after filtering)
Answer: SQL (Structured Query Language) is a language used to communicate with database servers. It serves as the front‑end for many database systems (MySQL, PostgreSQL, Oracle, Sybase, etc.) and is organized into three subsystems: data description, data access, and privileges. SQL is case‑sensitive, though keywords are typically written in upper case.
Sources: [{'source': 'sql.pdf', 'page': 3, 'score': 0.475327730178833, 'preview': 'Introduction to\nSQL What is SQL?\n▶ Structured Query Language\n▶ Usually “talk” to a database server\n▶ Used as front end to many databases (mysql,\npostgresql, oracle, sybase)\n▶ Three Subsystems: data description, data access and\nprivileges\n▶ Optimized for certain data arrangements\n▶ The language is ca...'}, {'source': 'sql.pdf', 'page': 5, 'score': 0.475327730178833, 'preview': 'Introduction to\nSQL What is SQL?\n▶ Structured Query Language\n▶ Usually “talk” to a database